# NB03 — Week 5: RUL Baselines + NASA Score + MLflow
**Objective:** RF & GBM on last-cycle features; RMSE + asymmetric NASA score;
runs tracked in MLflow under `aip/m2/rul-cmapss` with the frozen tag set.

Conversion target: `aip_signal/models.py`.

In [1]:
import json
from pathlib import Path

import mlflow
import numpy as np
import polars as pl
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor

DATA_DIR = Path("../../data")
meta = json.loads(next((DATA_DIR / "processed").glob("feature_meta_*.json")).read_text())
FD = next((DATA_DIR / "processed").glob("features_*_train.parquet")).stem.split("_")[1]
FEATS = meta["features"]
train = pl.read_parquet(DATA_DIR / f"processed/features_{FD}_train.parquet")
test = pl.read_parquet(DATA_DIR / f"processed/features_{FD}_test.parquet")

2026/09/19 01:18:15 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\vscode_hywork\dsml_projects\assetintelligenceplatform\aipenv\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


## 1. NASA score — late predictions penalised harder than early

In [2]:
def nasa_score(y, p):
    d = np.asarray(p, float) - np.asarray(y, float)
    return float(np.sum(np.exp(-d[d < 0] / 13) - 1) + np.sum(np.exp(d[d >= 0] / 10) - 1))


def rmse(y, p):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(p)) ** 2)))


assert nasa_score([100], [110]) > nasa_score([100], [90]) > 0  # asymmetry check

## 2. Train/eval — one prediction per test unit at its last observed cycle

In [3]:
x_tr, y_tr = train.select(FEATS).to_numpy(), train["rul"].to_numpy()
last = test.filter(pl.col("cycle") == pl.col("cycle").max().over("unit_id"))
x_te, y_te = last.select(FEATS).to_numpy(), last["rul"].to_numpy()

mlflow.set_tracking_uri(f"sqlite:///{(DATA_DIR / 'processed/mlflow_local.db').resolve()}")
mlflow.set_experiment("aip/m2/rul-cmapss")

MODELS = {
    "random_forest": RandomForestRegressor(n_estimators=120, min_samples_leaf=5,
                                           n_jobs=-1, random_state=42),
    "gradient_boosting": GradientBoostingRegressor(n_estimators=200, learning_rate=0.05,
                                                   max_depth=4, random_state=42),
}
results = {}
for name, model in MODELS.items():
    model.fit(x_tr, y_tr)
    pred = np.clip(model.predict(x_te), 0, None)
    m = {"rmse": rmse(y_te, pred), "nasa_score": nasa_score(y_te, pred)}
    with mlflow.start_run(run_name=f"{name}-{FD}"):
        mlflow.set_tags({"dataset": "cmapss", "fd_set": FD, "model_family": name,
                         "feature_version": meta["feature_version"]})
        mlflow.log_metrics(m)
        mlflow.log_dict({"features": FEATS}, "features.json")
    results[name] = m
    print(f"{name:20s} RMSE={m['rmse']:6.2f}  NASA={m['nasa_score']:8.1f}")

best = min(results, key=lambda k: results[k]["rmse"])
json.dump({"best_model": best, **results[best]},
          open(DATA_DIR / "processed/nb03_best.json", "w"), indent=2)
print("best:", best, "| NB03 complete.")

2026/09/19 01:18:19 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/09/19 01:18:19 INFO mlflow.store.db.utils: Updating database tables


2026/09/19 01:18:21 INFO mlflow.tracking.fluent: Experiment with name 'aip/m2/rul-cmapss' does not exist. Creating a new experiment.


2026/09/19 01:18:35 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



2026/09/19 01:18:35 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git branch is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



2026/09/19 01:18:35 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git repository URL is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



random_forest        RMSE= 12.50  NASA=    44.6


gradient_boosting    RMSE=  8.73  NASA=    22.8
best: gradient_boosting | NB03 complete.
